# 膝跳反射

In [7]:
from neuron import *
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

In [8]:
class Sigmoid:
    def __init__(self, alpha=1, beta=1):
        self.alpha = alpha
        self.beta = beta

    def __call__(self, x):
        return self.beta / (1 + np.exp(-self.alpha * x))

    def gradient(self, x):
        return self(x) * (1 - self(x) / self.beta) * self.alpha


def i_fn(t):
    sigmoid = Sigmoid(alpha=1.5, beta=55)
    if t < 10:
        return 0.0
    elif t < 20:
        return sigmoid.gradient(t - 15)
    else:
        return 0.0

In [9]:
t = np.arange(0, 50, 0.01)
i = np.array([i_fn(t_) for t_ in t])
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=i, mode='lines', name='Current'))
fig.show()

In [4]:
current_injector = CurrentInjector(i_fn)
afferent_neuron = Neuron(l=5000, d=10.0)    # 传入神经
synapse1 = Synapse(current_injector, afferent_neuron, delay=0.5)
motion_neuron1 = Neuron(l=4000, d=8.0)      # 运动神经 1
synapse2 = Synapse(afferent_neuron, motion_neuron1, g=0.4, e_syn=-65.0, delay=0.5)
inhibitory_neuron = Neuron(l=2000, d=8.0)   # 抑制性神经
synapse3 = Synapse(afferent_neuron, inhibitory_neuron, g=0.4, e_syn=-65.0, delay=0.5)
motion_neuron2 = Neuron(l=2000, d=8.0)      # 运动神经 2
synapse4 = Synapse(inhibitory_neuron, motion_neuron2, g=-0.4, e_syn=-65.0, delay=0.5)

recorder = Recorder(300, afferent_neuron, motion_neuron1, inhibitory_neuron, motion_neuron2)
recorder.run_network(synapse1, synapse2, synapse3, synapse4)

100%|██████████| 30000/30000 [00:09<00:00, 3300.85it/s]


In [5]:
data = []
neurons = [afferent_neuron, motion_neuron1, inhibitory_neuron, motion_neuron2]
for i in range(len(neurons)):
    neuron = neurons[i]
    for t in range(0, recorder.n_t, 200):
        for x in range(neuron.n_x):
            data.append({"neuron": i, "time": t, "x": neuron.dx * x, "v": recorder.v[i][x, t]})
df = pd.DataFrame(data)

In [6]:
fig = px.line(df, x="x", y="v", animation_frame="time", title="Voltage of Neurons", color="neuron", range_y=[-120, 40])
fig.show()